# GhostChef AI — Day 2: Data Merging & Master Dataset Construction

Build a single master dataset by merging sales, calendar, and price data.
Transform wide-format sales into long format, then enrich with temporal and pricing features.

**Memory note:** this version processes the wide sales table in row-chunks instead of building one
58M-row DataFrame in memory. Each chunk is melted, merged with calendar and sell prices, written to
Parquet, and then freed before the next chunk starts.

# Load Raw Data

**Objective:** Ingest all source CSV files into memory.

Load the five raw datasets: sales (validation & evaluation), calendar events, sell prices, and sample submission.

These source files are relatively small on their own — it is the *wide-to-long melt* of
`sales_train_validation` (1,913 day columns) that produces the ~58M-row table that no longer fits
comfortably in memory, so that step is the one we chunk below.

In [1]:
import pandas as pd
import numpy as np
import os
import gc
import math
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds


In [2]:
sell_prices = pd.read_csv('../data/raw/sell_prices.csv')
sample_submission = pd.read_csv('../data/raw/sample_submission.csv')
sales_train_validation = pd.read_csv('../data/raw/sales_train_validation.csv')
sales_train_evaluation = pd.read_csv('../data/raw/sales_train_evaluation.csv')
calendar = pd.read_csv('../data/raw/calendar.csv')


# Data Understanding

**Objective:** Inspect structure and schema of each dataset.

Preview the first rows of each source table to confirm columns, types, and granularity before merging.

In [3]:
sell_prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


In [4]:
sample_submission.head()

,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,...,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,HOBBIES_1_001_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,HOBBIES_1_002_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,HOBBIES_1_004_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,HOBBIES_1_005_CA_1_validation,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
sales_train_validation.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1904,d_1905,d_1906,d_1907,d_1908,d_1909,d_1910,d_1911,d_1912,d_1913
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,3,0,1,1,1,3,0,1,1
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,2,1,1,1,0,1,1,1
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,5,4,1,0,1,3,7,2
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,1,1,0,1,1,2,2,2,4


In [6]:
sales_train_evaluation.head()

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4,...,d_1932,d_1933,d_1934,d_1935,d_1936,d_1937,d_1938,d_1939,d_1940,d_1941
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,2,4,0,0,0,0,3,3,0,1
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,1,2,1,1,0,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,0,2,0,0,0,2,3,0,1
3,HOBBIES_1_004_CA_1_evaluation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,1,1,0,4,0,1,3,0,2,6
4,HOBBIES_1_005_CA_1_evaluation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0,...,0,0,0,2,1,0,0,2,1,0


In [7]:
calendar.head()

,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


# Wide to Long Transformation

**Objective:** Reshape the wide-format sales matrix into a tidy long-format table.

Each row in `sales_train_validation` has 1,913 day columns (d_1 … d_1913).
`pd.melt` unpivots these into one row per (item, day) pair, producing ~58M rows.

Instead of melting the entire table in one shot, `sales_train_validation` is split into row-chunks of
`CHUNK_SIZE` items. Each chunk is melted on its own, producing a much smaller long-format slice.

In [8]:
ID_COLUMNS = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id"
]


In [9]:
DAY_COLUMNS = sorted(
    (col for col in sales_train_validation.columns if col.startswith("d_")),
    key=lambda col: int(col.split("_", 1)[1])
)


# Merge with Calendar

**Objective:** Enrich each sales row with calendar features.

Join on `day` ↔ `d` to attach date, weekday, events, SNAP flags, and `wm_yr_wk`.
This adds 14 columns; the result is a 22-column dataframe at 58M rows (across all chunks combined).

This merge is now performed **inside the per-chunk loop**, immediately after each chunk is melted,
so only one chunk's worth of merged rows ever exists in memory at a time.

# Merge with Sell Prices

**Objective:** Attach the selling price for each (item, store, week) combination.

Join on `item_id` + `store_id` + `wm_yr_wk` to add `sell_price`.
Nulls appear when an item was not available for sale in a given week (~21% of rows).

This merge is also performed **inside the per-chunk loop**, right after the calendar merge, so each
chunk goes melt → merge calendar → merge sell_prices → save → free, before the next chunk starts.

# Chunk-Based Processing Pipeline

**Objective:** Build the master dataset without ever holding the full ~58M-row table in memory.

For every chunk of `CHUNK_SIZE` rows from `sales_train_validation`:
1. Melt only the current chunk (wide → long).
2. Merge the melted chunk with `calendar`.
3. Merge the result with `sell_prices`.
4. Save the processed chunk as a Parquet file (PyArrow).
5. Delete intermediate objects and run `gc.collect()` to free memory before the next chunk.

The final master dataset is the **collection of Parquet files** in `OUTPUT_DIR` — together they hold
exactly the same schema and values as the single `master_df` in the original notebook, just split
across files instead of held in one DataFrame.

In [10]:
CHUNK_SIZE = 3000  # number of item-rows (from sales_train_validation) melted per chunk
OUTPUT_DIR = "../data/interim/master_dataset_parquet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

n_rows = len(sales_train_validation)
n_chunks = math.ceil(n_rows / CHUNK_SIZE)

print(f"Total item-rows in sales_train_validation: {n_rows}")
print(f"Chunk size (rows melted at a time): {CHUNK_SIZE}")
print(f"Number of chunks to process: {n_chunks}")


Total item-rows in sales_train_validation: 30490
Chunk size (rows melted at a time): 3000
Number of chunks to process: 11


In [11]:
# Lightweight accumulators only — never a full copy of the master dataset.
total_long_rows = 0
null_counts_total = None
first_chunk_preview = None   # small head() snapshot from the first chunk, for inspection cells below

for i, start in enumerate(range(0, n_rows, CHUNK_SIZE), start=1):
    end = min(start + CHUNK_SIZE, n_rows)
    print(f"Processing chunk {i}/{n_chunks} (rows {start} to {end - 1})...")

    # 1. Melt only the current chunk
    chunk = sales_train_validation.iloc[start:end]

    melted = pd.melt(
        chunk,
        id_vars=ID_COLUMNS,
        value_vars=DAY_COLUMNS,
        var_name="day",
        value_name="sales"
    )
    del chunk

    # 2. Merge with calendar
    merged_calendar = melted.merge(
        calendar,
        left_on="day",
        right_on="d",
        how="left"
    )
    del melted

    # 3. Merge with sell_prices
    master_chunk = merged_calendar.merge(
        sell_prices,
        on=["item_id", "store_id", "wm_yr_wk"],
        how="left"
    )
    del merged_calendar

    # Capture one small preview (mirrors master_df.head() from the original notebook)
    if first_chunk_preview is None:
        first_chunk_preview = master_chunk.head().copy()

    # Track lightweight, aggregatable stats instead of keeping every chunk around
    total_long_rows += len(master_chunk)
    chunk_nulls = master_chunk.isnull().sum()
    null_counts_total = chunk_nulls if null_counts_total is None else null_counts_total.add(chunk_nulls, fill_value=0)

    # 4. Save the processed chunk as a Parquet file
    table = pa.Table.from_pandas(master_chunk, preserve_index=False)
    out_path = os.path.join(OUTPUT_DIR, f"chunk_{i:04d}.parquet")
    pq.write_table(table, out_path)
    print(f"  -> saved {len(master_chunk):,} rows to {out_path}")

    # 5. Free memory
    del master_chunk, table
    gc.collect()

print("All chunks processed.")
print(f"Total rows written across all chunk files: {total_long_rows:,}")


Processing chunk 1/11 (rows 0 to 2999)...
  -> saved 5,739,000 rows to master_dataset_parquet\chunk_0001.parquet
Processing chunk 2/11 (rows 3000 to 5999)...
  -> saved 5,739,000 rows to master_dataset_parquet\chunk_0002.parquet
Processing chunk 3/11 (rows 6000 to 8999)...
  -> saved 5,739,000 rows to master_dataset_parquet\chunk_0003.parquet
Processing chunk 4/11 (rows 9000 to 11999)...
  -> saved 5,739,000 rows to master_dataset_parquet\chunk_0004.parquet
Processing chunk 5/11 (rows 12000 to 14999)...
  -> saved 5,739,000 rows to master_dataset_parquet\chunk_0005.parquet
Processing chunk 6/11 (rows 15000 to 17999)...
  -> saved 5,739,000 rows to master_dataset_parquet\chunk_0006.parquet
Processing chunk 7/11 (rows 18000 to 20999)...
  -> saved 5,739,000 rows to master_dataset_parquet\chunk_0007.parquet
Processing chunk 8/11 (rows 21000 to 23999)...
  -> saved 5,739,000 rows to master_dataset_parquet\chunk_0008.parquet
Processing chunk 9/11 (rows 24000 to 26999)...
  -> saved 5,739,00

In [12]:
# master_df.shape equivalent — no single object of this shape ever exists in memory,
# but the combined Parquet dataset has exactly this shape.
n_columns = len(first_chunk_preview.columns)
print((total_long_rows, n_columns))


(58327370, 23)


In [13]:
# master_df.head() equivalent — head of the first processed chunk
first_chunk_preview


,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,...,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0,2011-01-29,11101,...,2011,d_1,NaN,NaN,NaN,NaN,0,0,0,NaN


In [14]:
# master_df.info() equivalent, using the tracked totals + dtypes from the preview chunk
print(f"Represented across {n_chunks} Parquet chunk files in '{OUTPUT_DIR}'")
print(f"Total entries: {total_long_rows:,}")
print(f"Data columns (total {n_columns} columns):")
first_chunk_preview.dtypes


Represented across 11 Parquet chunk files in 'master_dataset_parquet'
Total entries: 58,327,370
Data columns (total 23 columns):


id               object
item_id          object
dept_id          object
cat_id           object
store_id         object
state_id         object
day              object
sales             int64
date             object
wm_yr_wk          int64
weekday          object
wday              int64
month             int64
year              int64
d                object
event_name_1     object
event_type_1     object
event_name_2     object
event_type_2     object
snap_CA           int64
snap_TX           int64
snap_WI           int64
sell_price      float64
dtype: object

In [15]:
# master_df.isnull().sum() equivalent — nulls accumulated chunk-by-chunk during the loop above
null_counts_total.astype(int)


id                     0
item_id                0
dept_id                0
cat_id                 0
store_id               0
state_id               0
day                    0
sales                  0
date                   0
wm_yr_wk               0
weekday                0
wday                   0
month                  0
year                   0
d                      0
event_name_1    53631910
event_type_1    53631910
event_name_2    58205410
event_type_2    58205410
snap_CA                0
snap_TX                0
snap_WI                0
sell_price      12299413
dtype: int32

# Validation

**Objective:** Verify data integrity of the final master dataset.

Check for duplicate rows and review numeric column statistics.
Event columns have expected nulls (most days are not events); `sell_price` nulls indicate items not yet available for sale.

Since the master dataset now lives as Parquet chunk files rather than one in-memory DataFrame, the
checks below read back only the columns they need (via PyArrow) instead of loading everything at once.

In [16]:
# master_df.duplicated().sum() equivalent — an exact check, limited to the ('id', 'day') key
# columns, since that's already enough to detect duplicate rows here without holding the
# full 22-column dataset in memory. Reads straight from the saved Parquet files, so this
# cell is independent of the chunk-processing loop above (safe to rerun after a kernel restart).
key_dataset = ds.dataset(OUTPUT_DIR, format="parquet")
key_df = key_dataset.to_table(columns=["id", "day"]).to_pandas()

duplicate_count = key_df.duplicated().sum()
print(f"Duplicate (id, day) pairs across the full dataset: {duplicate_count}")

del key_dataset, key_df
gc.collect()


Duplicate (id, day) pairs across the full dataset: 0


0

In [17]:
# master_df.describe() equivalent — read only the numeric columns back from the saved Parquet
# dataset (via PyArrow), rather than re-loading the full 22-column table. Column names come
# from the dataset's own schema, so this cell doesn't depend on the loop cell having run first.
dataset = ds.dataset(OUTPUT_DIR, format="parquet")

numeric_cols = [c for c in ["sales", "sell_price", "wm_yr_wk", "snap_CA", "snap_TX", "snap_WI"]
                if c in dataset.schema.names]

numeric_table = dataset.to_table(columns=numeric_cols)
numeric_df = numeric_table.to_pandas()
description = numeric_df.describe()

del numeric_table, numeric_df
gc.collect()

description


,sales,sell_price,wm_yr_wk,snap_CA,snap_TX,snap_WI
count,5.832737e+07,4.602796e+07,5.832737e+07,5.832737e+07,5.832737e+07,5.832737e+07
mean,1.126322e+00,4.408089e+00,1.133919e+04,3.293257e-01,3.293257e-01,3.293257e-01
std,3.873108e+00,3.403657e+00,1.503742e+02,4.699684e-01,4.699684e-01,4.699684e-01
min,0.000000e+00,1.000000e-02,1.110100e+04,0.000000e+00,0.000000e+00,0.000000e+00
25%,0.000000e+00,2.180000e+00,1.121700e+04,0.000000e+00,0.000000e+00,0.000000e+00
50%,0.000000e+00,3.470000e+00,1.133300e+04,0.000000e+00,0.000000e+00,0.000000e+00
75%,1.000000e+00,5.840000e+00,1.144800e+04,1.000000e+00,1.000000e+00,1.000000e+00
max,7.630000e+02,1.073200e+02,1.161300e+04,1.000000e+00,1.000000e+00,1.000000e+00


# Save Final Master Dataset

**Objective:** Persist the merged dataset to disk as Parquet chunk files.

Unlike the original notebook (a single CSV written from one in-memory `master_df`), the save step
already happened **inside the chunk-processing loop** above — step 4 of each iteration. This section
just confirms what was written.

In [18]:
saved_files = sorted(os.listdir(OUTPUT_DIR))
total_bytes = sum(os.path.getsize(os.path.join(OUTPUT_DIR, f)) for f in saved_files)

print(f"Number of Parquet files saved: {len(saved_files)}")
print(f"Total size on disk: {total_bytes / (1024 ** 2):.2f} MB")
saved_files[:5]


Number of Parquet files saved: 11
Total size on disk: 99.40 MB


['chunk_0001.parquet',
 'chunk_0002.parquet',
 'chunk_0003.parquet',
 'chunk_0004.parquet',
 'chunk_0005.parquet']

In [19]:
# master_df["sales"].head() equivalent
first_chunk_preview["sales"].head()


0    0
1    0
2    0
3    0
4    0
Name: sales, dtype: int64